# House Price Prediction – End-to-End Machine Learning

**Dataset:** `data.csv`  
**Problem type:** Supervised Machine Learning – Regression  
**Target column:** `price`

## Project objective
Build a machine learning model to estimate house prices using property characteristics such as bedrooms, bathrooms, living area, lot size, floors, condition, year built, and location.

## Workflow
1. Import libraries and load the dataset
2. Understand and inspect the data
3. Clean data and prepare features
4. Perform exploratory data analysis (EDA)
5. Split data into training and testing sets
6. Build numerical and categorical preprocessing pipelines
7. Train regression models
8. Evaluate and compare model performance
9. Save and reload the best model
10. Make a sample prediction

**Note:** This notebook follows the workflow of the supplied reference notebook. Since `price` is numeric and continuous, Logistic Regression and accuracy score are replaced with regression models and regression metrics.

## 1. Import required libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib
import warnings
warnings.filterwarnings("ignore")

## 2. Load the dataset
Keep `data.csv` in the same folder as this notebook. In Google Colab, upload the CSV before running this cell.

In [ ]:
df = pd.read_csv("data.csv")
df.head()

## 3. Understand the dataset

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

## 4. Data cleaning and feature preparation
Convert the date column to datetime and extract year, month, and day. The original date and street columns are removed from model features. The street field is a highly specific address and can create many unique categories.

In [ ]:
# Convert date to datetime
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Extract useful date features
df["sale_year"] = df["date"].dt.year
df["sale_month"] = df["date"].dt.month
df["sale_day"] = df["date"].dt.day

# Remove rows where the target price is missing, if any
df = df.dropna(subset=["price"]).copy()

# Remove exact duplicate rows
df = df.drop_duplicates().copy()

df.head()

In [ ]:
df.shape

## 5. Exploratory Data Analysis (EDA)

In [ ]:
# Distribution of house prices
plt.figure(figsize=(8, 5))
sns.histplot(df["price"], kde=True)
plt.title("Distribution of House Prices")
plt.xlabel("Price")
plt.show()

In [ ]:
# Price distribution without extreme values for easier visualization
upper_limit = df["price"].quantile(0.99)
plt.figure(figsize=(8, 5))
sns.histplot(df.loc[df["price"] <= upper_limit, "price"], kde=True)
plt.title("House Prices (up to 99th percentile)")
plt.xlabel("Price")
plt.show()

In [ ]:
# Average price by number of bedrooms
bedroom_price = df.groupby("bedrooms")["price"].mean().sort_index()
bedroom_price.plot(kind="bar", figsize=(8, 5))
plt.title("Average House Price by Bedrooms")
plt.xlabel("Bedrooms")
plt.ylabel("Average Price")
plt.show()

In [ ]:
# Correlation between numerical variables
numeric_for_corr = df.select_dtypes(include=["int64", "float64"])
plt.figure(figsize=(12, 8))
sns.heatmap(numeric_for_corr.corr(), cmap="coolwarm", annot=False)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# Average price by city (top 15 cities by number of records)
top_cities = df["city"].value_counts().head(15).index
city_prices = (df[df["city"].isin(top_cities)]
               .groupby("city")["price"].mean()
               .sort_values(ascending=False))
city_prices.plot(kind="bar", figsize=(12, 5))
plt.title("Average House Price – Top 15 Cities by Record Count")
plt.ylabel("Average Price")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 6. Define input features (X) and target (y)
The target is `price`. The original `date` and `street` fields are not used. Location fields `city`, `statezip`, and `country` are retained as categorical features.

In [ ]:
X = df.drop(columns=["price", "date", "street"])
y = df["price"]

X.head()

In [ ]:
y.head()

In [ ]:
print("Features shape:", X.shape)
print("Target shape:", y.shape)

## 7. Train-test split
Use 80% of the data for training and 20% for testing. `random_state=42` makes the split reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

## 8. Identify numerical and categorical columns

In [ ]:
numerical_columns = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
numerical_columns

In [ ]:
categorical_columns = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
categorical_columns

## 9. Create preprocessing pipelines
Numerical columns: fill missing values with the median and scale values. Categorical columns: fill missing values with the most frequent value and apply one-hot encoding.

In [ ]:
numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

numerical_pipeline

In [ ]:
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehotencoder", OneHotEncoder(handle_unknown="ignore"))
])

categorical_pipeline

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ("num", numerical_pipeline, numerical_columns),
    ("cat", categorical_pipeline, categorical_columns)
])

preprocessor

## 10. Train a Linear Regression model

In [ ]:
linear_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

linear_model

In [ ]:
linear_model.fit(X_train, y_train)

In [ ]:
linear_predictions = linear_model.predict(X_test)
linear_predictions[:10]

## 11. Evaluate the Linear Regression model
For regression, use MAE, RMSE, and R². Lower MAE/RMSE indicates smaller prediction errors; an R² closer to 1 indicates that the model explains more variation in the target.

In [ ]:
linear_mae = mean_absolute_error(y_test, linear_predictions)
linear_rmse = np.sqrt(mean_squared_error(y_test, linear_predictions))
linear_r2 = r2_score(y_test, linear_predictions)

print("Linear Regression Results")
print("MAE :", round(linear_mae, 2))
print("RMSE:", round(linear_rmse, 2))
print("R2  :", round(linear_r2, 4))

## 12. Train a Random Forest Regression model

In [ ]:
random_forest_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

random_forest_model

In [ ]:
random_forest_model.fit(X_train, y_train)

In [ ]:
rf_predictions = random_forest_model.predict(X_test)
rf_predictions[:10]

## 13. Evaluate the Random Forest model

In [ ]:
rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))
rf_r2 = r2_score(y_test, rf_predictions)

print("Random Forest Results")
print("MAE :", round(rf_mae, 2))
print("RMSE:", round(rf_rmse, 2))
print("R2  :", round(rf_r2, 4))

## 14. Compare both models

In [ ]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "MAE": [linear_mae, rf_mae],
    "RMSE": [linear_rmse, rf_rmse],
    "R2 Score": [linear_r2, rf_r2]
})

results.sort_values(by="RMSE")

## 15. Select and save the best model
This selects the model with the lower RMSE on the test set. The saved pipeline includes preprocessing, so new input data receives the same transformations.

In [ ]:
if rf_rmse < linear_rmse:
    best_model = random_forest_model
    best_model_name = "Random Forest"
else:
    best_model = linear_model
    best_model_name = "Linear Regression"

print("Selected model:", best_model_name)

joblib.dump(best_model, "house_price_model.pkl")
print("Model saved as house_price_model.pkl")

## 16. Load the saved model and verify predictions

In [ ]:
loaded_model = joblib.load("house_price_model.pkl")
loaded_predictions = loaded_model.predict(X_test)

print("First 10 predictions:", loaded_predictions[:10])

## 17. Actual vs predicted prices

In [ ]:
comparison = pd.DataFrame({
    "Actual Price": y_test.values,
    "Predicted Price": loaded_predictions
})

comparison.head(10)

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, loaded_predictions, alpha=0.5)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted House Prices")
plt.tight_layout()
plt.show()

## 18. Make a prediction for a new house
The example below uses an existing test row as sample input. You can replace its values with details for a different house, keeping the same feature columns.

In [ ]:
sample_house = X_test.iloc[[0]].copy()

predicted_price = loaded_model.predict(sample_house)[0]
print("Predicted House Price:", round(predicted_price, 2))
print("Actual House Price  :", round(y_test.iloc[0], 2))

## 19. Project conclusion

- The dataset contains 4,600 house records and 18 original columns.
- The target variable is `price`.
- Date-derived features are created from the sale date.
- Numerical and categorical features are processed using separate pipelines.
- Linear Regression and Random Forest Regression are trained and compared.
- MAE, RMSE, and R² are used to evaluate model performance.
- The model with the lower test RMSE is saved as `house_price_model.pkl`.

**Important:** Model scores depend on the train-test split and feature choices. Review the evaluation results before using the model for real-world pricing. This is a learning project, not a professional property valuation.